# Voxel 4D timeseries

Port of `packages/niivue/examples/vox.4d.html`. Loads the ASL timeseries with the first five frames and exposes frame, graph, calibration, colorbar, and pixel-ratio controls.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"


def hex_to_rgba(value):
    value = value.lstrip("#")
    return [int(value[i : i + 2], 16) / 255 for i in (0, 2, 4)] + [1.0]


volume_id = "mpld_asl"

nv = NiiVue(
    background_color=[0.1, 0.1, 0.1, 1.0],
    is_graph_visible=True,
    is_colorbar_visible=True,
    show_render=1,
    slice_type=3,
    backend="webgl2",
)

slice_type = widgets.Dropdown(
    options=[("Axial", 0), ("Coronal", 1), ("Sagittal", 2), ("A+C+S+R", 3), ("Render", 4)],
    value=3,
    description="Slice",
)
frame = widgets.IntSlider(value=0, min=0, max=4, step=1, description="Frame")
graph = widgets.Checkbox(value=True, description="Graph")
normalize = widgets.Checkbox(value=False, description="Normalize")
fixed_range = widgets.Checkbox(value=False, description="Fixed range")
auto_cal = widgets.Checkbox(value=False, description="Auto cal")
colorbar = widgets.Checkbox(value=True, description="Colorbars")
background = widgets.ColorPicker(value="#191923", description="Background")
pixel_ratio = widgets.Dropdown(
    options=[("Auto", -1.0), ("1.0", 1.0), ("1.5", 1.5), ("2.0", 2.0)],
    value=-1.0,
    description="Pixel",
)


def update_slice(change):
    nv.slice_type = change["new"]


def update_frame(change):
    nv.set_frame_4d(volume_id, change["new"])
    if auto_cal.value:
        nv.recalculate_cal_min_max(0, change["new"])


def update_graph(change):
    nv.is_graph_visible = change["new"]


def update_normalize(change):
    nv.graph_normalize_values = change["new"]


def update_fixed_range(change):
    nv.graph_is_range_cal_min_max = change["new"]


def update_colorbar(change):
    nv.is_colorbar_visible = change["new"]


def update_background(change):
    nv.background_color = hex_to_rgba(change["new"])


def update_pixel_ratio(change):
    nv.device_pixel_ratio = change["new"]


slice_type.observe(update_slice, names="value")
frame.observe(update_frame, names="value")
graph.observe(update_graph, names="value")
normalize.observe(update_normalize, names="value")
fixed_range.observe(update_fixed_range, names="value")
colorbar.observe(update_colorbar, names="value")
background.observe(update_background, names="value")
pixel_ratio.observe(update_pixel_ratio, names="value")

controls = widgets.VBox(
    [
        widgets.HBox([slice_type, frame, background, pixel_ratio]),
        widgets.HBox([graph, normalize, fixed_range, auto_cal, colorbar]),
    ]
)
display(controls)
display(nv)

nv.load_volumes(
    [
        {
            "url": f"{VOLUMES}/mpld_asl.nii.gz",
            "name": volume_id,
            "id": volume_id,
            "limitFrames4D": 5,
        }
    ]
)